**This problem set is due Wednesday, November 25, 2026 at 11:59 pm. Please plan ahead and submit your work on time.**

# Problem Set 09: Gibbs sampling and Hidden Markov Models (HMMs)


In this problem set, you will explore Gibbs sampling and Hidden Markov Models (HMMs) by implementing some foundational algorithms.

0. [Credit for Contributors (required)](#contributors)

1. [Sampling (25 points)](#problem1)
  
2. [HMMs (60 points)](#problem2)
   1. [Conditioning Warmup (5 points)](#conditioning)
   2. [HMM State (5 points)](#hmm-state)
   3. [Transition Distribution Creation (5 points)](#transition)
   4. [Observation Variable Creation (5 points)](#observation-variable)
   5. [Observation CPT creation (20 points)](#observation-cpt)
   6. [Viterbi Decoding (20 points)](#viterbi)

3. [Adversarial Conditions for Rejection Sampling (10 points)](#problem3)
   1. [Designing Your Bayes Net [AI-enabled] (7 points)](#rejection)
   2. [Explaining Your Bayes Net (3 points)](#rejection-explanation)

4. [Time Spent on Pset (5 points)](#part4)
    
**100 points** total for Problem Set 9

## Imports and Utilities

In [1]:
# Be sure to run the cell below to import the code needed for this assignment.
from __future__ import division

%matplotlib inline
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import copy
from collections import namedtuple
import itertools

# imports for autograder
from principles_of_autonomy.grader import Grader
from principles_of_autonomy.notebook_tests.pset_9 import TestPSet9

## <a id="contributors"></a> 0. Credit for Contributors

List the various students, lecture notes, or online resouces that helped you complete this problem set:

Ex: I worked with Bob on the cat chase HMM problem.

<div class="alert alert-info">
Write your answer in the cell below this one.
</div>

--> *(double click on this cell to delete this text and type your answer here)*

## <a name="problem1"></a> 1. Sampling (25 points)

Consider the following 3 node factor graph:

<center>
<img src="3node_Gibbs.png" width=500>
</center>

The table on the right gives three different CPT tables. Gibbs sampling did a poor job on generating samples that reflected the joint of these three variables for one of these CPTs. Which table was it, 1, 2 or 3?

In [2]:
## Enter your answer by changing the assignment expression below, e.g., q1_answer = 3
q1_answer = 2

In [3]:
# test_1
Grader.run_single_test_inline(TestPSet9, "test_01", locals())

Test passed!!

.

----------------------------------------------------------------------
Ran 1 test in 0.001s

OK


Consider the following Bayes net.

<center>
<img src="sampling_bayes.png" width=50%>
</center>

We are interested in drawing samples from the distribution described by this network, conditioned on evidence, using prior sampling with rejection.

Is this a good way to sample from $P(K \mid B = b)$?

In [4]:
## Enter your answer by changing the assignment expression below, e.g., q2_answer = 'no' or q2_answer = 'yes'
q2_answer = 'yes'

In [5]:
# test_2
Grader.run_single_test_inline(TestPSet9, "test_02", locals())

Test passed!!

.

----------------------------------------------------------------------
Ran 1 test in 0.001s

OK


Is this a good way to sample from $P(B \mid K = k)$?

In [6]:
## Enter your answer by changing the assignment expression below, e.g., q3_answer = 'no' or q3_answer = 'yes'
q3_answer = 'no'

In [7]:
# test_3
Grader.run_single_test_inline(TestPSet9, "test_03", locals())

Test passed!!

.

----------------------------------------------------------------------
Ran 1 test in 0.001s

OK


Enter the set of nodes in the Markov Blanket of node G, e.g. `['B','D']`.

In [8]:
## Enter your answer as a tuple of strings, e.g., q4_answer = ('B', 'D')
q4_answer = ('D', 'I')

In [9]:
# test_4
Grader.run_single_test_inline(TestPSet9, "test_04", locals())

Test passed!!

.

----------------------------------------------------------------------
Ran 1 test in 0.001s

OK


Enter the set of nodes in the Markov Blanket of node D, e.g. `['B','D']`.

In [10]:
## Enter your answer as a tuple of strings, e.g., q5_answer = ('B', 'D')
q5_answer = ('B', 'E', 'G', 'H')

In [11]:
# test_5
Grader.run_single_test_inline(TestPSet9, "test_05", locals())

Test passed!!

.

----------------------------------------------------------------------
Ran 1 test in 0.001s

OK


## <a name="problem2"></a> 2. HMMs (60 points)

Let's define some utility classes and methods first, for our random variables and our conditional probability tables. 

In [12]:
class RV:
    '''A random variable with a finite domain.

    Example usage:
      A = RV("A", ["x", "y", "z"])
      print(A.domain)
      print(A.dim)
      B = RV("B", [(0, 0), (0, 1), (0, 2)]))
      print(B.domain)
      print(B.dim)
    '''

    def __init__(self, name, domain):
        '''Initialize a RV.

        Args:
          name: str name for the RV.
          domain: list or tuple of domain values.
        '''
        assert isinstance(domain, (list, tuple))
        self.name = name
        self.domain = domain
        self.dim = len(domain)

    def __hash__(self):
        return hash((self.name, tuple(self.domain)))

    def __eq__(self, other):
        return self.name == other.name and self.domain == other.domain

    def __repr__(self):
        return f"RV('{self.name}', {self.domain})"

class CPT:
    '''A CPT over RVs.

    Example usage:
      A = RV("varA", ["x", "y", "z"])
      B = RV("varB", [0, 1])
      table = np.array([
        [0.1, 0.0],
        [0.4, 0.9],
        [0.5, 0.1]
      ])
      cpt = CPT([A, B], table)
      print(cpt.rvs)
      print(cpt.get(("y", 0)))
      print(cpt.get_by_rvs({A: "y", B: 0}))
      print(cpt.get_by_names({"varA": "y", "varB": 0}))
    '''

    def __init__(self, rvs, table):
        '''Create a cpt from a list of RVs and a numpy array.

        The order of the random variables corresponds to the axes
        of the numpy array.

        Args:
          rvs: A list or tuple of RVs.
          array: A numpy array of the probabilities.

        Returns:
          cpt: A CPT.'''
        assert isinstance(rvs, (tuple, list))
        assert len(rvs) == len(table.shape)
        assert all(rv.dim == dim for (rv, dim) in zip(rvs, table.shape))
        assert isinstance(table, np.ndarray)
        self.rvs = rvs
        self.table = table

    def set(self, assignment, new_value):
        '''Given a complete assignment and a value, update table.

        Args:
          assignment: A tuple of values in the order of self.rv.
          new_value: A new value to add to the table.

        Returns:
          value: The value in self.table.
        '''
        assert len(assignment) == len(self.rvs)
        indices = [None for _ in self.rvs]
        for index, value in enumerate(assignment):
            rv = self.rvs[index]
            indices[index] = rv.domain.index(value)
        self.table[tuple(indices)] = new_value

    def get(self, assignment):
        '''Given a complete assignment of values, lookup table value.

        Args:
          assignment: A tuple of values in the order of self.rv.

        Returns:
          value: The value in self.table.
        '''
        assert len(assignment) == len(self.rvs)
        indices = [None for _ in self.rvs]
        for index, value in enumerate(assignment):
            rv = self.rvs[index]
            indices[index] = rv.domain.index(value)
        return self.table[tuple(indices)]

    def get_by_rvs(self, rvs_to_vals):
        '''Given a complete assignment of RVs to values, lookup table value.

        Args:
          rvs_to_values: A dict from RVs to values in their domains.

        Returns:
          value: The value in self.table.
        '''
        assert set(rvs_to_vals.keys()) == set(self.rvs)
        indices = [None for _ in self.rvs]
        for rv, value in rvs_to_vals.items():
            index = self.rvs.index(rv)
            indices[index] = rv.domain.index(value)
        return self.table[tuple(indices)]

    def get_by_names(self, rv_name_dict):
        '''Given a dict from RV names (strs) to assignments,
        return the corresponding value in the CPT table.

        Args:
          rv_name_dict: A dict from str names to values.
          cpt: A CPT.

        Returns:
          value: The float value from cpt.table.
        '''
        assert len(rv_name_dict) == len(self.rvs)
        rv_name_to_rv = {rv.name: rv for rv in self.rvs}
        rvs_to_vals = {}
        for rv_name, value in rv_name_dict.items():
            rv = rv_name_to_rv[rv_name]
            rvs_to_vals[rv] = value
        return self.get_by_rvs(rvs_to_vals)

    def __hash__(self):
        return hash(tuple(self.rvs)) ^ hash(self.table.tobytes())

    def __eq__(self, other):
        return hash(self) == hash(other)

    def __neq__(self, other):
        return not (self == other)

    def allclose(self, other, decimals=6):
        '''Check whether two CPTs are (nearly) equal.
        '''
        if set(self.rvs) != set(other.rvs):
            raise ValueError("Can only compare CPTs with the same RVs.")
        new_idxs = [other.rvs.index(rv) for rv in self.rvs]
        trans_table2 = np.transpose(other.table, new_idxs)
        assert self.table.shape == trans_table2.shape
        return np.allclose(self.table, trans_table2)


def condition_CPT(cpt, evidence):
    """Given a CPT and an assignment of one or more RVs as evidence,
        create a new posterior CPT representing the effect of the evidence.

    Example usage:
        A, B = RV('A', ["x", "y", "z"]), RV('B', [0, 1])
        cpt = CPT([A, B],
            np.array([
                [1, 0],
                [0, 1],
                [0, 0],
        ]))
        evidence = {A: "y"}
        result = posterior_CPT(cpt, evidence)
        assert result.rvs == [B]
        assert result.get_by_rvs({B: 0}) == 0
        assert result.get_by_rvs({B: 1}) == 1

    Args:
        cpt: A CPT.
        evidence: A dict from RVs to values.

    Returns:
        posterior_CPT: A CPT.
    """
    new_rvs = [rv for rv in cpt.rvs if rv not in evidence]
    idxs = []
    for rv in cpt.rvs:
        if rv in evidence:
            value = evidence[rv]
            idx = rv.domain.index(value)
        else:
            idx = slice(None)
        idxs.append(idx)
    new_table = cpt.table[tuple(idxs)].copy()
    return CPT(new_rvs, new_table)


def normalize_CPT(cpt):
    """Normalize the values of a CPT so they sum to 1.

    Args:
        cpt: A CPT.

    Returns:
        new_cpt: A CPT.
    """
    denom = cpt.table.sum()
    if denom == 0:
        raise Exception("WARNING: tried to normalize an all-zeros CPT.")
    return CPT(cpt.rvs, cpt.table / denom)


def max_CPT(cpt, rv=None):
    """Create a new CPT where rv has been maximized out.

    If rv is None, then this function returns the max
    over all assignments in the CPT.

    Example usage:
        A, B = RV('A', ["x", "y", "z"]), RV('B', [0, 1])
        cpt = CPT([A, B],
            np.array([
                [1., 5.],
                [2., 6.],
                [3., 4.],
        ]))
        assert max_CPT(cpt) == 6.
        cpt1 = max_CPT(cpt, rv=A)
        assert cpt1.rvs == [B]
        assert cpt1.get_by_rvs({B: 0}) == 3
        assert cpt1.get_by_rvs({B: 1}) == 6
        cpt2 = max_CPT(cpt, rv=B)
        assert cpt2.rvs == [A]
        assert cpt2.get_by_rvs({A: "x"}) == 5
        assert cpt2.get_by_rvs({A: "y"}) == 6
        assert cpt2.get_by_rvs({A: "z"}) == 4

    Args:
        cpt: A CPT.
        rv: A random variables in the CPT or None.

    Returns:
        new_cpt: A CPT.
    """
    if rv is None:
        return cpt.table.max()
    axis = cpt.rvs.index(rv)
    new_table = cpt.table.max(axis=axis)
    new_rvs = list(cpt.rvs)
    new_rvs.remove(rv)
    return CPT(new_rvs, new_table)


def argmax_CPT(cpt, rv=None):
    """Create a new "CPT" where the values are domain assignments
    of the given rv that lead to maximum values.

    If rv is None, then this function returns the argmax assignment
    over all assignments in the CPT.

    Example usage:
        A, B = RV('A', ["x", "y", "z"]), RV('B', [0, 1])
        cpt = CPT([A, B],
            np.array([
                [1., 5.],
                [2., 0.],
                [3., 4.],
        ]))
        assert argmax_CPT(cpt) == ("x", 1)
        cpt1 = argmax_CPT(cpt, rv=A)
        assert cpt1.rvs == [B]
        assert cpt1.get_by_rvs({B: 0}) == "z"
        assert cpt1.get_by_rvs({B: 1}) == "x"
        cpt2 = argmax_CPT(cpt, rv=B)
        assert cpt2.rvs == [A]
        assert cpt2.get_by_rvs({A: "x"}) == 1
        assert cpt2.get_by_rvs({A: "y"}) == 0
        assert cpt2.get_by_rvs({A: "z"}) == 1

    Args:
        cpt: A CPT.
        rv: A random variable in the CPT or None.

    Returns:
        new_cpt: A CPT.
    """
    if rv is None:
        idxs = np.unravel_index(cpt.table.argmax(), cpt.table.shape)
        return tuple(v.domain[idx] for v, idx in zip(cpt.rvs, idxs))
    axis = cpt.rvs.index(rv)
    new_table_idxs = cpt.table.argmax(axis=axis)
    new_table = np.empty(new_table_idxs.shape, dtype=object)
    for idx, val in np.ndenumerate(new_table_idxs):
        new_table[idx] = rv.domain[val]
    new_rvs = list(cpt.rvs)
    new_rvs.remove(rv)
    return CPT(new_rvs, new_table)


HMM = namedtuple("HMM", [
    "state_rvs",  # A list of RVs, one per timestep
    "observation_rvs",  # A list of RVs, one per timestep
    "transition_CPTs",  # A list of CPTs, one per transition
    "observation_CPTs",  # A list of CPTs, one per timestep
    "initial_distribution"  # A single CPT
])

def iter_joint_values(rvs):
    '''Iterates over joint assignments for a list of RVs.

    Returns an iterator that can be used in a for loop.

    Example usage:
      for assignment in iter_joint_values(rvs):
        print(assignment)  # a tuple
        assert assignment[0] in rvs[0].domain

    Args:
      rvs: A list of RVs.

    Yields:
      assignment: A tuple of ints representing a joint
        assignment of the random variables.
    '''
    domains = [rv.domain for rv in rvs]
    return itertools.product(*domains)

def get_sub_assignment(rvs, assignment, sub_rvs):
    '''Given an assignment of rvs to values, get a subassignment,
    that is, a sub-tuple of the given assignment involving only
    the given sub_rvs.

    Example usage:
      x = RV("x", [0, 1])
      y = RV("y", ["a", "b"])
      z = RV("z", [3, 5])
      rvs = (x, y, z)
      assignment = (0, "b", 3)
      sub_rvs = (z, x)
      sub_assignment = get_sub_assignment(rvs, assignment, sub_rvs)
      assert sub_assignment == (3, 0)

    Args:
      rvs: A tuple or list of RVs.
      assignment: A tuple or list of values.
      sub_rvs: A tuple or list of RVs, a subset of rvs.

    Returns:
      sub_assignment: A tuple of values.
    '''
    assert set(sub_rvs).issubset(set(rvs))
    sub_assignment = []
    for rv in sub_rvs:
        idx = rvs.index(rv)
        val = assignment[idx]
        sub_assignment.append(val)
    return tuple(sub_assignment)

def multiply_CPTs(cpts):
    '''Multiply CPTs together.

    Args:
      cpts: A list of CPTs.

    Returns:
      result: A new CPT.
    '''
    # Note: this implementation favors clarity over
    # efficiency. If you are curious about a more
    # efficient implementation, please ask course staff :)

    # Collect all random variables
    rvs = set()
    for t in cpts:
        rvs |= set(t.rvs)
    rvs = list(rvs)

    # Initialize result table
    table = np.empty([rv.dim for rv in rvs])

    # Initialize result CPT
    result = CPT(rvs, table)

    # Iterate over assignments in the new CPT
    for assignment in iter_joint_values(rvs):
        value = 1.
        for t in cpts:
            # Get the sub-assignment
            sub_assignment = get_sub_assignment(rvs, assignment, t.rvs)
            # Compute product
            value *= t.get(sub_assignment)
        # Add value to result table
        result.set(assignment, value)

    # Finalize CPT
    return result


def create_hmm(obstacle_map, num_timesteps, noise_prob=0.):
    """Converts a map into an HMM.

    Creates the state RVs, the observation RVs, the transition CPTs,
    and the observation CPTs.

    You should not need to use this; it's used in tests.

    Args:
      obstacle_map: A list of lists of ints; see example and
          description in `create_state_variable`.
      num_timesteps: An int number of timesteps, must be >= 1.
      noise_prob: Observation noise parameter.s

    Returns:
      problem: An HMM.
    """
    assert num_timesteps >= 1

    # Create variables
    state_rvs = []
    observation_rvs = []
    for t in range(num_timesteps):
        state_t = create_state_variable(obstacle_map, f"state{t}")
        obs_t = create_observation_variable(f"observation{t}")
        state_rvs.append(state_t)
        observation_rvs.append(obs_t)

    # Create observation CPTs
    observation_CPTs = []
    for state, obs in zip(state_rvs, observation_rvs):
        obs_cpt_t = create_observation_cpt(obstacle_map, state, obs,
                                                 noise_prob=noise_prob)
        observation_CPTs.append(obs_cpt_t)

    # Create transition CPTs
    transition_CPTs = []
    if num_timesteps > 1:
        for state_t, state_t1 in zip(state_rvs[:-1], state_rvs[1:]):
            trans_cpt_t = create_transition_cpt(obstacle_map, state_t, state_t1)
            transition_CPTs.append(trans_cpt_t)

    # Create (uniform) initial state distribution
    init_state_rv = state_rvs[0]
    table = np.ones((init_state_rv.dim,)) / init_state_rv.dim
    initial_distribution = CPT([init_state_rv], table)

    return HMM(state_rvs, observation_rvs, transition_CPTs,
               observation_CPTs, initial_distribution)

In this section, we will implement an HMM for a robot that is moving around randomly in a 2D grid with obstacles. The robot has sensors that allow it to detect obstacles in its immediate vicinity. It knows the grid map, with the locations of all obstacles, but it is uncertain about its own location in the grid. We will use Viterbi to determine the most likely locations for the robot given a sequence of local and potentially noisy observations.

Concretely, we will represent the 2D grid with obstacles as a list of lists, where 1s represent obstacles and 0s represent free space. Example:

```
obstacle_map = [
  [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0],
  [1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1],
  [1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0],
  [0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0]
]
```

The state of the robot is its location in the grid, represented as a tuple of ints, (row, col). Transitions are uniformly distributed amongst the robot's **current location** and the neighboring free (not obstacle) locations, where neighboring = 4 cardinal directions (up, down, left, right).

Observations are a 4-tuple that list which directions have obstacles, in order [N E S W], with a 1 for an obstacle and 0 for no obstacle. Observations that are "off the map" are 1, as though they are obstacles. For instance, in the map above, if there were no observation noise, then the observation for the top left corner (state=(0, 0)) would be (1, 0, 1, 1). Observations can also be corrupted with noise; see the `create_observation_cpt` docstring for more details.

Our ultimate task will be to take in a sequence of observations and return the corresponding sequence of most likely states.

### <a name="conditioning"></a> 2A. Conditioning Warmup (5 points)

Given a CPT and a variable that occurs in it, return a new CPT conditioned on the specified variable having value 0.

For reference, our solution is **1** line(s) of code.

In [13]:
def conditioning_warmup(cpt, rv):
    '''Given a CPT and a variable that occurs in it, return a new
    CPT conditioned on the specified variable having value 0.

    Args:
      cpt: CPT
      rv: A RV in cpt that has 0 in its domain.

    Returns:
      new_cpt: CPT
    '''
    return condition_CPT(cpt, {rv: 0})

In [14]:
# test_6
Grader.run_single_test_inline(TestPSet9, "test_06", locals())

Test passed!!

.

----------------------------------------------------------------------
Ran 1 test in 0.001s

OK


### <a name="hmm-state"></a> 2B. HMM State (5 points)


Write a function that creates a random variable for a state at a given time step in an obstacle HMM. The domain of the state variable should be a list of (row, col) indices into the map. Only free positions (not obstacles) should be included in the domain of the state variable. See docstring for more description.

For reference, our solution is **2** line(s) of code.

In [15]:
def create_state_variable(obstacle_map, name):
    '''Creates a RV for the HMM state.

    The domain of the state variable is a list of (row, col) indices
    for all free positions (0s) in row-major order.
    '''
    domain = [(r, c) for r, row in enumerate(obstacle_map) for c, val in enumerate(row) if val == 0]
    return RV(name, domain)

In [16]:
# test_7
Grader.run_single_test_inline(TestPSet9, "test_07", locals())

Test passed!!

.

----------------------------------------------------------------------
Ran 1 test in 0.001s

OK


### <a name="transition"></a> 2C. Transition Distribution Creation (5 points)


Write a function that creates a CPT for the transition distribution between states $s_{t}$ and $s_{t+1}$. Refer to the previous question for more information about the state variables and their domains.

For reference, our solution is **10** line(s) of code.

In [17]:
def create_transition_cpt(obstacle_map, st, st1):
    '''Create a CPT for the transition from state s_t to s_{t+1}.

    Transitions are uniformly distributed amongst the robot's current
    location and the neighboring free (not obstacle) locations (4 cardinal directions).
    '''
    table = np.zeros((st.dim, st1.dim))
    for i, (prev_r, prev_c) in enumerate(st.domain):
        for j, (next_r, next_c) in enumerate(st1.domain):
            if abs(prev_r - next_r) + abs(prev_c - next_c) <= 1:
                table[i, j] = 1.0
        table[i] /= table[i].sum()
    return CPT([st, st1], table)

In [18]:
# test_8
Grader.run_single_test_inline(TestPSet9, "test_08", locals())

Test passed!!

.

----------------------------------------------------------------------
Ran 1 test in 0.001s

OK


### <a name="observation-variable"></a> 2D. Observation Variable Creation (5 points)


Write a function that creates a random variable for an observation at a given time step in an obstacle HMM. See docstring for description.

For reference, our solution is **2** line(s) of code.

In [19]:
def create_observation_variable(name):
    '''Creates a RV for the HMM observation with the given name.

    Domain is all 16 4-tuples in {0, 1}^4 corresponding to [N, E, S, W].
    '''
    domain = list(itertools.product([0, 1], repeat=4))
    return RV(name, domain)

In [20]:
# test_9
Grader.run_single_test_inline(TestPSet9, "test_09", locals())

Test passed!!

.

----------------------------------------------------------------------
Ran 1 test in 0.001s

OK


### <a name="observation-cpt"></a> 2E. Observation CPT Creation (20 points)


Write a function that creates a CPT for the observation distribution between $s_{t}$ and $z_{t}$.

For reference, our solution is **26** line(s) of code.


In [21]:
def create_observation_cpt(obstacle_map, state_rv, observation_rv,
                           noise_prob=0.):
    '''Create a CPT P(observation_rv | state_rv) for the map.

    Directions are [N, E, S, W]; off-map counts as obstacle (1).
    Each entry flips independently with probability noise_prob.
    '''
    num_rows = len(obstacle_map)
    num_cols = len(obstacle_map[0])
    dirs = [(-1, 0), (0, 1), (1, 0), (0, -1)]  # N, E, S, W
    table = np.zeros((state_rv.dim, observation_rv.dim))
    for i, (r, c) in enumerate(state_rv.domain):
        true_obs = tuple(
            1 if (r + dr < 0 or r + dr >= num_rows or c + dc < 0 or c + dc >= num_cols)
            else obstacle_map[r + dr][c + dc]
            for dr, dc in dirs
        )
        for j, obs in enumerate(observation_rv.domain):
            flips = sum(o != t for o, t in zip(obs, true_obs))
            table[i, j] = (noise_prob ** flips) * ((1.0 - noise_prob) ** (4 - flips))
    return CPT([state_rv, observation_rv], table)

In [22]:
# test_10
Grader.run_single_test_inline(TestPSet9, "test_10", locals())

Test passed!!

.

----------------------------------------------------------------------
Ran 1 test in 0.001s

OK


### <a name="viterbi"></a> 2F. Viterbi Decoding (20 points)


Complete the implementation of Viterbi decoding. Note: in general, it is a good idea for numerical stability to implement Viterbi in log space. However, for the purpose of this homework, we are not expecting your implementation to be in log space. Moreover, the provided helper functions (e.g., `multiply_CPTs`) assume that you are _not_ working in log space.

For reference, our solution is **23** line(s) of code after comments are removed.

In [23]:
def run_viterbi(hmm, observations):
    '''Run the Viterbi algorithm to compute the most likely state
    assignments given observations.

    Normalizes messages after each step to avoid numerical underflow.
    '''
    T = len(observations)
    obs0 = condition_CPT(hmm.observation_CPTs[0], {hmm.observation_rvs[0]: observations[0]})
    msg = normalize_CPT(multiply_CPTs([hmm.initial_distribution, obs0]))
    backpointers = []

    for t in range(1, T):
        obs_t = condition_CPT(hmm.observation_CPTs[t], {hmm.observation_rvs[t]: observations[t]})
        joint = multiply_CPTs([msg, hmm.transition_CPTs[t - 1], obs_t])
        backpointers.append(argmax_CPT(joint, rv=hmm.state_rvs[t - 1]))
        msg = normalize_CPT(max_CPT(joint, rv=hmm.state_rvs[t - 1]))

    best_state = argmax_CPT(msg)[0]
    most_likely_states = [best_state]
    for t in range(T - 2, -1, -1):
        best_state = backpointers[t].get_by_rvs({hmm.state_rvs[t + 1]: best_state})
        most_likely_states.append(best_state)
    most_likely_states.reverse()
    return most_likely_states

In [24]:
# test_11
Grader.run_single_test_inline(TestPSet9, "test_11", locals())

Test passed!!

.

----------------------------------------------------------------------
Ran 1 test in 0.011s

OK


## <a name="problem3"></a> 3. Adversarial Conditions for Rejection Sampling (10 points)

### <a name="rejection"></a> 3A. Designing Your Bayes Net [AI-enabled] (7 points)

Rejection sampling estimates $P(\cdot \mid \mathbf{E} = \mathbf{e})$ by drawing complete samples from the Bayes net's prior (forward sampling) and throwing away every sample that disagrees with the evidence. On average, the algorithm keeps a fraction of samples equivalent to the likelihood of the evidence $P(\mathbf{E} = \mathbf{e})$. 

One might guess that rejection sampling is safe as long as each piece of evidence is individually likely. Your task is to show that this guess is wrong. Using an LLM (Claude, ChatGPT, etc.) build a Bayes net and an evidence assignment such that:

1. The net has at most 10 variables, each binary with domain `[0, 1]`.
2. Every CPT entry is in [0.05, 0.95] (no deterministic relationships).
3. The evidence assigns values to at most 3 variables.
4. Each evidence value is likely on its own: $P(E_i = e_i) \geq 0.5$ for every evidence variable $E_i$.
5. The evidence as a whole is unlikely: $P(\mathbf{E} = \mathbf{e}) \leq 0.005$.

Represent your net as a list of `CPT`s (the classes from Problem 2), one per variable. In each `CPT`, list the variable's parents first and the variable itself last, so `table[..., :]` sums to 1 over the last axis (like in `create_transition_cpt`). Represent the evidence as a dict mapping variable names to values.

Grading: The autograder forward-samples your net 8,000 times with a fixed random seed, marks each sample ACCEPT or REJECT, and reports the total empirical acceptance rate. You pass if at most 80 samples (1%) are accepted. It also checks conditions 1–4 exactly. If your net truly satisfies condition 5, the chance of failing because of an unlucky draw is below one in 100 million. 

In [25]:
def rejection_sampling_adversary():
    '''Returns:
      cpts: list of CPT, one per variable (parents first, child last).
      evidence: dict mapping variable name (str) -> value in {0, 1}.
    '''
    E1 = RV("E1", [0, 1])
    E2 = RV("E2", [0, 1])
    E3 = RV("E3", [0, 1])

    # P(E1): P(E1=1) = 0.51
    cpt_e1 = CPT([E1], np.array([0.49, 0.51]))

    # P(E2 | E1): strongly anti-correlated with E1
    # P(E2=1) = 0.49 * 0.95 + 0.51 * 0.07 = 0.5012
    cpt_e2 = CPT([E1, E2], np.array([
        [0.05, 0.95],
        [0.93, 0.07],
    ]))

    # P(E3 | E1, E2): noisy NAND gate
    # P(E3=1 | E1=1, E2=1) = 0.05; otherwise 0.95 -> P(E3=1) = 0.91787
    # Joint P(E1=1, E2=1, E3=1) = 0.51 * 0.07 * 0.05 = 0.001785 <= 0.005
    cpt_e3 = CPT([E1, E2, E3], np.array([
        [[0.05, 0.95], [0.05, 0.95]],
        [[0.05, 0.95], [0.95, 0.05]],
    ]))

    cpts = [cpt_e1, cpt_e2, cpt_e3]
    evidence = {"E1": 1, "E2": 1, "E3": 1}
    return cpts, evidence

In [26]:
# test_12
Grader.run_single_test_inline(TestPSet9, "test_12_rejection_adversary", locals())

ValueError: no such test method in <class 'principles_of_autonomy.notebook_tests.pset_9.TestPSet9'>: test_12_rejection_adversary

In [27]:
# Local verification for Problem 3A (exact conditions 1-5 + 8,000 forward samples)
cpts_3a, ev_3a = rejection_sampling_adversary()
assert len(cpts_3a) <= 10 and len(ev_3a) <= 3
for cpt in cpts_3a:
    assert all(rv.domain == [0, 1] for rv in cpt.rvs)
    assert np.all(cpt.table >= 0.05) and np.all(cpt.table <= 0.95)
    assert np.allclose(cpt.table.sum(axis=-1), 1.0)
joint_3a = multiply_CPTs(cpts_3a)
p_joint = joint_3a.get_by_names(ev_3a)
print(f"Exact joint P(E=e) = {p_joint:.6f} (must be <= 0.005)")
assert p_joint <= 0.005
for k, v in ev_3a.items():
    rv = [r for r in joint_3a.rvs if r.name == k][0]
    ax = joint_3a.rvs.index(rv)
    other = tuple(i for i in range(len(joint_3a.rvs)) if i != ax)
    marg = joint_3a.table.sum(axis=other)[rv.domain.index(v)]
    print(f"Exact marginal P({k}={v}) = {marg:.6f} (must be >= 0.5)")
    assert marg >= 0.5
rng = np.random.default_rng(0)
accepted = 0
for _ in range(8000):
    sample = {}
    for cpt in cpts_3a:
        idx = tuple(p.domain.index(sample[p.name]) for p in cpt.rvs[:-1])
        sample[cpt.rvs[-1].name] = int(rng.choice(cpt.rvs[-1].domain, p=cpt.table[idx]))
    if all(sample[k] == v for k, v in ev_3a.items()):
        accepted += 1
print(f"Empirical accepted out of 8,000 samples: {accepted} ({accepted/8000:.3%}, must be <= 80)")
assert accepted <= 80
print("All Problem 3A checks passed!")

Exact joint P(E=e) = 0.001785 (must be <= 0.005)
Exact marginal P(E1=1) = 0.510000 (must be >= 0.5)
Exact marginal P(E2=1) = 0.501200 (must be >= 0.5)
Exact marginal P(E3=1) = 0.917870 (must be >= 0.5)
Empirical accepted out of 8,000 samples: 9 (0.112%, must be <= 80)
All Problem 3A checks passed!


### <a name="rejection-explanation"></a> 3B. Explaining Your Bayes Net (3 points)

**This part is _not_ AI-enabled.** Write your explanation yourself, without the help of an AI assistant.

In the cell below, explain how your solution to 3A works. Why is the evidence as a whole so unlikely? Please point to the specific structure in your net that is responsible.

<div class="alert alert-info">
Write your answer in the cell below this one.
</div>

--> *(double click on this cell to delete this text and type your answer here)*

# <a name="part4"></a> Time Spent on Pset (5 points)

Please use [this form](https://forms.gle/RHH3Ukc6Ns1qVYJQ6) to tell us how long you spent on this pset. After you submit the form, the form will give you a confirmation word. Please enter that confirmation word below to get the 5 points. 

In [28]:
form_confirmation_word = "INSERT FORM CONFIRMATION WORD HERE"

In [29]:
# Run all tests
Grader.grade_output([TestPSet9], [locals()], "results.json")
Grader.print_test_results("results.json")

Test passed!!

Test passed!!

Test passed!!

Test passed!!

Test passed!!

Test passed!!

... (2 outputs left out)

Test passed!!

Test passed!!

Test passed!!

Total score is 100/105.

Score for test_01 (principles_of_autonomy.notebook_tests.pset_9.TestPSet9.test_01) is 5/5.

Score for test_02 (principles_of_autonomy.notebook_tests.pset_9.TestPSet9.test_02) is 5/5.

Score for test_03 (principles_of_autonomy.notebook_tests.pset_9.TestPSet9.test_03) is 5/5.

Score for test_04 (principles_of_autonomy.notebook_tests.pset_9.TestPSet9.test_04) is 5/5.

Score for test_05 (principles_of_autonomy.notebook_tests.pset_9.TestPSet9.test_05) is 5/5.

Score for test_06 (principles_of_autonomy.notebook_tests.pset_9.TestPSet9.test_06) is 5/5.

Score for test_07 (principles_of_autonomy.notebook_tests.pset_9.TestPSet9.test_07) is 5/5.

Score for test_08 (principles_of_autonomy.notebook_tests.pset_9.TestPSet9.test_08) is 10/10.

Score for test_09 (principles_of_autonomy.notebook_tests.pset_9.TestPSet9.test_09) is 5/5.

Score for test_10 (principles_of_autonomy.notebook_tests.pset_9.TestPSet9.test_10) is 25/25.

Score for test_11 (principles_of_autonomy.notebook_

In [30]:
###############################################
#   SAVE YOUR NOTEBOOK BEFORE SUBMITTING!!!   #
###############################################